# RGCNN Implementation Subject-dependency

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import numpy as np
import pandas as pd
import glob as gb
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from DGCNN_Data_Model import PrecomputedEEGDataset, DGCNNBlock 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
folder_path = '../SEEDiv_processed/filtered_csv' #filtered_csv'
batch_size = 64
channel = 62
feature_bands = 5
num_classes = 4
num_epochs = 50
lr = 0.002
Adj_K = 4

#### Full Training Loop

In [ ]:
def accuracy(preds, labels):
    pred_labels = torch.argmax(preds, dim=1)
    return (pred_labels == labels).float().mean().item()

In [ ]:
def train_dgcnn(model, train_loader, val_loader):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0001)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        total_acc = 0

        for x_batch, label_dict in train_loader:
            x_batch = x_batch.to(device)              # [B, 5, 62]
            labels = label_dict['emotion'].to(device) # [B]
            logits = model(x_batch)                     # [B, num_classes]

            loss = loss_fn(logits, labels)
            acc = accuracy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_acc += acc

        avg_loss = total_loss / len(train_loader)
        avg_acc = total_acc / len(train_loader)

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_loss:.4f} | Train Acc: {avg_acc:.4f}")

        # Optional: evaluate on validation set
        if val_loader is not None:
            val_acc = evaluate_dgcnn(model, val_loader, device)
            print(f"→ Validation Acc: {val_acc:.4f}")


### Cross-session Evaluation Loop

In [ ]:
@torch.no_grad()
def evaluate_dgcnn(model, data_loader):
    model.eval()
    total_acc = 0

    for x_batch, label_dict in data_loader:
        x_batch = x_batch.to(device)
        labels = label_dict['emotion'].to(device)
        x_avg = x_batch.mean(dim=1)

        logits = model(x_avg)
        acc = accuracy(logits, labels)
        total_acc += acc

    return total_acc / len(data_loader)

In [ ]:
def get_cross_session_folds(dataset, subject_id):
    # Group sample indices by session for this subject
    session_to_indices = defaultdict(list)

    for idx in range(len(dataset)):
        _, label = dataset[idx]
        if label['subject'] == subject_id:
            session = label['session']
            session_to_indices[session].append(idx)

    folds = []
    sessions = sorted(session_to_indices.keys())
    for test_session in sessions:
        test_idx = session_to_indices[test_session]
        train_idx = [i for s in sessions if s != test_session for i in session_to_indices[s]]
        folds.append((train_idx, test_idx, test_session))

    return folds  # list of (train_idx, test_idx, test_session_id)

In [ ]:
def cross_session_evaluation(dataset):
    subject_ids = sorted(set(dataset[i][1]['subject'] for i in range(len(dataset))))
    results = []

    for subject in subject_ids:
        print(f"\n Subject {subject}")
        folds = get_cross_session_folds(dataset, subject)

        for fold_id, (train_idx, test_idx, test_session) in enumerate(folds):
            train_set = Subset(dataset, train_idx)
            test_set = Subset(dataset, test_idx)

            train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
            test_loader = DataLoader(test_set, batch_size=16)

            model = DGCNNBlock(in_features=5, hidden_features=64, num_classes=4, K=Adj_K, num_nodes=62)
            train_dgcnn(model, train_loader, val_loader=None)

            acc = evaluate_dgcnn(model, test_loader)

            print(f"Fold {fold_id+1}: Train on S{[i for i in range(1,4) if i != test_session]} → Test on S{test_session} | Acc: {acc:.4f}")
            results.append({
                "subject": subject,
                "test_session": test_session,
                "accuracy": acc,
                "num_epochs": num_epochs,
                "hidden_dim": 32,
                "cheb_K": 4,
                "lr": lr
            })

    return results

In [ ]:
folder_path = "../SEEDiv_processed/precomputed"
ouput_path = "DGCNNN_CrossSession_results"
os.makedirs(ouput_path, exist_ok=True)
features_list = [file for file in sorted(gb.glob(os.path.join(folder_path,'*'))) if file.endswith('.pt')]
labels_list =  [file for file in sorted(gb.glob(os.path.join(folder_path,'*'))) if file.endswith('.csv')]

dataset = PrecomputedEEGDataset(features_list,labels_list)
results = cross_session_evaluation(dataset)
df = pd.DataFrame(results)
df.to_csv(os.path.join(ouput_path,"loso_dgcnn_results.csv"), index=False)